In [ ]:
import time
from jetbot import Robot
from math import sqrt

from dstarlite import DStarLiteController, Wall
from NOD import ObstacleDetector

# ---------- Konfiguration ----------
GRID_W, GRID_H = 20, 20
start = (0, 0)
goal  = (15, 15)

ROTATE_SPEED = 0.18
FORWARD_SPEED = 0.22
SECONDS_PER_CELL = 0.5    # grov: tid for at køre én grid-celle

OBSTACLE_FRAMES_REQUIRED = 2

# ---------- Init hardware + algoritmer ----------
robot = Robot()
detector = ObstacleDetector()
detector.start()   # start object detection i baggrundstråd

controller = DStarLiteController(width=GRID_W, height=GRID_H, start=start, goal=goal)

grid_x, grid_y = start
robot_heading = 0  # initial heading

# Brug D*'s egen metode til initiale kommandoer
command_queue = controller.get_initial_commands()
command_index = 0

obstacle_counter = 0
replanned_on_this_obstacle = False


# ---------- Hjælpefunktioner ----------
def cell_step_for_heading(heading_deg):
    """Returner (dx, dy) for 8 retninger baseret på heading (afrundet til nærmeste 45°)."""
    # normalisér til [-180, 180]
    h = ((heading_deg + 180) % 360) - 180
    # rund til nærmeste 45°
    q = round(h / 45) * 45

    if q == 0:
        return (0, 1)      # frem i +y
    elif q == 45:
        return (1, 1)      # diagonal ned-højre
    elif q == 90:
        return (1, 0)      # højre
    elif q == 135:
        return (1, -1)     # diagonal op-højre
    elif q in (180, -180):
        return (0, -1)     # op i -y
    elif q == -135:
        return (-1, -1)    # diagonal op-venstre
    elif q == -90:
        return (-1, 0)     # venstre
    elif q == -45:
        return (-1, 1)     # diagonal ned-venstre
    else:
        # fallback
        return (0, 0)

def cell_in_front(grid_x, grid_y, robot_heading):
    dx, dy = cell_step_for_heading(robot_heading)
    return grid_x + dx, grid_y + dy

    #if robot_heading == 0:
     #   return grid_x, grid_y + 1
    
   # if robot_heading == 45:
    #    return grid_x + 1, grid_y + 1
    
    #if robot_heading == 90:
     #   return grid_x + 1, grid_y
    
    #if robot_heading == 135:
     #   return grid_x + 1, grid_y -1
    
    #if robot_heading == -45:
     #   return grid_x - 1, grid_y + 1
    
    #if robot_heading == -90:
     #   return grid_x - 1, grid_y
    
    #if robot_heading == -135:
      #  return grid_x - 1, grid_y - 1*/

def execute_command(angle_deg, dist_cells):
    global robot_heading, grid_x, grid_y

    ROTATE_SPEED = 0.18
    FORWARD_SPEED = 0.22
    SECONDS_PER_CELL = 0.5

    # ---------- Drej ----------
    if angle_deg < 0:
        robot.left(ROTATE_SPEED)
    elif angle_deg > 0:
        robot.right(ROTATE_SPEED)

    # tid proportional med vinkel (groft)
    time.sleep(abs(angle_deg) / 270.0)
    robot.stop()

    # opdater heading
    robot_heading += angle_deg
    if robot_heading > 180:
        robot_heading -= 360
    if robot_heading < -180:
        robot_heading += 360

    # ---------- Kør frem ----------
    # hvor mange celler? √2 ≈ 1.41 → 1 step, 2.0 → 2 step osv.
    steps = max(1, dist_cells)

    move_time = steps * SECONDS_PER_CELL
    robot.forward(FORWARD_SPEED)
    time.sleep(move_time)
    robot.stop()

    # ---------- Opdater grid-position ----------
    dx, dy = cell_step_for_heading(robot_heading)
        
    grid_x += dx
    grid_y += dy

    print(f"Ny grid-position: ({grid_x}, {grid_y}), heading={robot_heading}")
    controller.update_robot_position((grid_x, grid_y))

    # sync heading ind i controller for næste replan
    try:
        controller.robot_heading = robot_heading
    except AttributeError:
        pass



# ---------- Hovedløkke ----------
try:
    while True:
        # Læs vision-tilstand fra baggrundstråden
        obstacle_close, frac, dist_cm, frame = detector.get_state()
        print(f"vision: close={obstacle_close}, frac={frac:.2f}, dist={dist_cm}")

        # Opdater tæller for "stabil" forhindring
        if obstacle_close:
            obstacle_counter += 1
        else:
            obstacle_counter = 0
            replanned_on_this_obstacle = False  # klar til næste forhindring

        # Hvis vi har set en forhindring flere frames i træk → replanner ÉN gang
        if obstacle_counter > OBSTACLE_FRAMES_REQUIRED and frac > 0.8 and not replanned_on_this_obstacle:
            print("Ny forhindring – opdaterer D*")
            robot.stop()

            fx, fy = cell_in_front(grid_x, grid_y, robot_heading)
            print(f"Robot grid pos: ({grid_x}, {grid_y}), heading={robot_heading}")
            print(f"Wall at: ({fx}, {fy})")

            wall = Wall(fx, fy, fx, fy)
            new_commands = controller.handle_obstacle(wall)
            print("Nye kommandoer fra D*:", new_commands)

            if new_commands:
                command_queue = new_commands
                command_index = 0
                replanned_on_this_obstacle = True
            else:
                print("D* fandt ingen rute – jeg bliver stående, men kører videre i loop.")

        # Følg D*-kommandoer, hvis der er nogen
        if command_index < len(command_queue):
            angle, dist = command_queue[command_index]
            print("COMMAND:", angle, dist)
            execute_command(angle, dist)
            time.sleep(1)
            command_index += 1
        else:
            # ingen flere kommandoer → enten mål nået eller midlertidig ingen rute
            robot.stop()
            print("Ingen flere kommandoer – venter på evt. ny forhindring eller ny rute.")
            # vi breaker IKKE – loopen fortsætter, så vi stadig ser objekter
            time.sleep(0.2)

        time.sleep(0.1)

except KeyboardInterrupt:
    pass
finally:
    robot.stop()
    detector.stop()
    print("Stopper.")


Kalibrerer gulv... fjern objekter foran robotten.
Kalibrering færdig.
vision: close=False, frac=0.00, dist=None
COMMAND: 0 1.0
Ny grid-position: (0, 1), heading=0
vision: close=True, frac=0.82, dist=7.313640312771502
COMMAND: 0 1.0
Ny grid-position: (0, 2), heading=0
vision: close=True, frac=0.81, dist=7.410211267605634
COMMAND: 0 1.0
Ny grid-position: (0, 3), heading=0
vision: close=True, frac=0.77, dist=7.823420074349442
COMMAND: 0 1.0
Ny grid-position: (0, 4), heading=0
vision: close=True, frac=0.75, dist=7.971590909090908
COMMAND: 0 1.0
Ny grid-position: (0, 5), heading=0
vision: close=True, frac=0.77, dist=7.7442502299908
Ingen flere kommandoer – venter på evt. ny forhindring eller ny rute.
vision: close=True, frac=0.94, dist=6.367624810892587
Ny forhindring – opdaterer D*
Robot grid pos: (0, 5), heading=0
Wall at: (0, 6)
Nye kommandoer fra D*: [(0, 1.0)]
COMMAND: 0 1.0
Ny grid-position: (0, 6), heading=0
vision: close=True, frac=0.84, dist=7.1764705882352935
Ingen flere kommandoe